# Phase 9 -- Advanced Optimization for Score Improvement

Based on the deep research analysis, this notebook implements:

1. **Logit Adjustment (LA) Loss** -- Replaces Focal Loss; mathematically aligned with Macro F1 under class imbalance
2. **Adversarial Weight Perturbation (AWP)** -- Flat minima regularization to prevent overfitting on 80-sample minority classes
3. **DeBERTa-v3-base 5-fold retraining** with LA Loss + AWP
4. **ModernBERT-base 5-fold training** -- Second transformer backbone for structural/formatting detection
5. **Enhanced feature engineering** -- POS bigrams, char n-gram SVD, burstiness, pseudo-perplexity
6. **Distribution-Aware Soft Pseudo-Labeling** -- Safe test set signal extraction
7. **Improved ensemble** -- 5+ model diversity, Ridge meta-learner, per-class thresholds

Previous best: OOF F1 = 0.9591, Public LB = 0.92170

In [ ]:
# ============================================================
# PHASE 9: ADVANCED OPTIMIZATION -- KAGGLE T4x2 VERSION
# ============================================================
# This notebook is SELF-CONTAINED. Only needs train.csv and test.csv.
# No external src/ files required.
# ============================================================
import sys
sys.modules['cupy'] = None  # Hack to prevent dask/lightgbm from trying to load broken cupy

import xgboost as xgb
import lightgbm as lgb
import os, warnings, gc, time, json, re, string, sys
import numpy as np
import pandas as pd  
from collections import Counter
from scipy import sparse
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

import xgboost as xgb
import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

np.random.seed(SEED)
torch.manual_seed(SEED)

gc.collect()

# ---- DEVICE SETUP (works on Kaggle CUDA, local MPS, or CPU) ----
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.empty_cache()
    USE_AMP = True   # FP16 mixed precision on T4
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    
    # Safe fallback if properties can't be fetched
    try:
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU Memory: {mem:.1f} GB')
    except:
        pass
        
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    USE_AMP = False
else:
    DEVICE = torch.device('cpu')
    USE_AMP = False

# ---- DATA PATHS (auto-detect Kaggle vs local) ----
KAGGLE_PATHS = [
    '/kaggle/input/datasets/aliivaezii/test-and-train',   # your custom dataset upload
    '/kaggle/input/malto-recruitment-hackathon',          # default competition data
    '/kaggle/input/test-and-train',                       
    '/kaggle/input',                                      
]

DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p
        break

if DATA_DIR is None:
    # Local fallback
    for p in ['.', '..', '/Users/aliivaezii/Documents/MALTO']:
        if os.path.exists(os.path.join(p, 'train.csv')):
            DATA_DIR = p
            break

assert DATA_DIR is not None, "Cannot find train.csv! Check your data upload."
print(f'Data directory: {DATA_DIR}')

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# CRITICAL FIX: Fill NaN texts to prevent loss=nan crashes during training
train_df['TEXT'] = train_df['TEXT'].fillna("empty")
test_df['TEXT'] = test_df['TEXT'].fillna("empty")

y_all = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test = test_df['TEXT'].values

# Class priors for Logit Adjustment
class_counts = np.bincount(y_all, minlength=NUM_LABELS)
class_priors = class_counts / class_counts.sum()

# Output directories
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('submissions', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print(f'Device: {DEVICE} | AMP: {USE_AMP}')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Class counts: {dict(enumerate(class_counts))}')
print(f'Class priors: {dict(enumerate(np.round(class_priors, 4)))}')

Device: mps
Train: (2400, 2), Test: (600, 2)
Class counts: {0: np.int64(1520), 1: np.int64(80), 2: np.int64(160), 3: np.int64(80), 4: np.int64(240), 5: np.int64(320)}
Class priors: {0: np.float64(0.6333), 1: np.float64(0.0333), 2: np.float64(0.0667), 3: np.float64(0.0333), 4: np.float64(0.1), 5: np.float64(0.1333)}


## 9A: Loss Functions -- Logit Adjustment and LDAM

**Why replace Focal Loss?**

Focal Loss was designed for dense object detection in CV -- it down-weights easy negatives.
But in our text classification, DeepSeek vs Grok is an *ambiguous* boundary, not easy vs hard.
Focal Loss amplifies noise on these borderline samples.

**Logit Adjustment (LA) Loss** subtracts `tau * log(prior_probability)` from the logits before
computing cross-entropy. This forces the network to produce larger logits for rare classes
(DeepSeek, Claude) to achieve the same loss value as the majority class.

In [ ]:
# ============================================================
# LOSS FUNCTIONS (NaN-safe)
# ============================================================

class LogitAdjustmentLoss(nn.Module):
    """Logit Adjustment Loss for long-tailed classification.
    
    Adjusts logits by subtracting tau * log(class_prior) before CE.
    
    IMPORTANT: Do NOT pass class_weights here. LA already handles
    class imbalance via log-prior shift. Combining both causes
    double-correction -> gradient explosion -> NaN loss.
    """
    def __init__(self, class_priors, tau=0.5, label_smoothing=0.0):
        super().__init__()
        log_priors = torch.log(torch.tensor(class_priors, dtype=torch.float32) + 1e-12)
        self.register_buffer('log_priors', log_priors)
        self.tau = tau
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        adjusted = logits - self.tau * self.log_priors.to(logits.device)
        # Clamp for numerical safety
        adjusted = adjusted.clamp(-30, 30)
        return F.cross_entropy(adjusted, targets, label_smoothing=self.label_smoothing)


class LDAMLoss(nn.Module):
    """Label-Distribution Aware Margin Loss with capped DRW weights."""
    def __init__(self, class_counts, max_margin=0.5, drw_epoch=2, total_epochs=6):
        super().__init__()
        margins = max_margin / np.power(class_counts, 0.25)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch = drw_epoch
        self.current_epoch = 0
        
        # DRW weights -- capped at 5x minimum to prevent explosion
        inv_freq = 1.0 / class_counts
        raw_w = inv_freq / inv_freq.sum() * len(class_counts)
        raw_w = np.clip(raw_w, raw_w.min(), raw_w.min() * 5)
        raw_w = raw_w / raw_w.sum() * len(class_counts)
        self.drw_weights = torch.tensor(raw_w, dtype=torch.float32)
    
    def set_epoch(self, epoch):
        self.current_epoch = epoch
    
    def forward(self, logits, targets):
        margins_for_batch = self.margins.to(logits.device)[targets]
        adjusted_logits = logits.clone()
        adjusted_logits[torch.arange(len(targets)), targets] -= margins_for_batch
        
        if self.current_epoch >= self.drw_epoch:
            weight = self.drw_weights.to(logits.device)
        else:
            weight = None
        
        return F.cross_entropy(adjusted_logits, targets, weight=weight)


# Quick tests
la_loss = LogitAdjustmentLoss(class_priors, tau=0.5)
dummy_logits = torch.randn(4, 6)
dummy_targets = torch.tensor([0, 1, 2, 5])
print(f'LA Loss test: {la_loss(dummy_logits, dummy_targets).item():.4f}')

# Verify no NaN even with extreme logits
extreme_logits = torch.randn(4, 6) * 50
assert not torch.isnan(la_loss(extreme_logits, dummy_targets)), "NaN detected!"
print(f'LA Loss (extreme logits): {la_loss(extreme_logits, dummy_targets).item():.4f} -- OK')

ldam_loss = LDAMLoss(class_counts.astype(float))
print(f'LDAM Loss test: {ldam_loss(dummy_logits, dummy_targets).item():.4f}')

print(f'\nLA adjustments (tau=0.5):')
for c in range(NUM_LABELS):
    adj = 0.5 * la_loss.log_priors[c].item()
    print(f'  {LABEL_MAP[c]:10s} (prior={class_priors[c]:.4f}): adjustment={adj:+.4f}')

LA Loss test: 2.7576
LDAM Loss test: 2.4028

LDAM margins per class:
  Human      (n=1520): margin=0.0801
  DeepSeek   (n=  80): margin=0.1672
  Grok       (n= 160): margin=0.1406
  Claude     (n=  80): margin=0.1672
  Gemini     (n= 240): margin=0.1270
  ChatGPT    (n= 320): margin=0.1182

LA log-prior adjustments (tau=1.0):
  Human      (prior=0.6333): adjustment=-0.4568
  DeepSeek   (prior=0.0333): adjustment=-3.4012
  Grok       (prior=0.0667): adjustment=-2.7081
  Claude     (prior=0.0333): adjustment=-3.4012
  Gemini     (prior=0.1000): adjustment=-2.3026
  ChatGPT    (prior=0.1333): adjustment=-2.0149


## 9B: Adversarial Weight Perturbation (AWP)

AWP injects noise directly into model **weights** (not inputs) to find flat minima.
This prevents the transformer from memorizing the 80 DeepSeek samples.

Key protocol:
- Activated only after warmup (epoch >= 1)
- Perturbation bounded by `adv_lr` and `adv_eps`
- Only upper 4 encoder layers + classifier head are perturbed

In [3]:
# ============================================================
# ADVERSARIAL WEIGHT PERTURBATION (AWP)
# ============================================================

class AWP:
    """Adversarial Weight Perturbation for flat minima optimization.
    
    Perturbs upper encoder layers + classifier head weights adversarially
    to regularize training and prevent overfitting on small classes.
    """
    def __init__(self, model, adv_lr=1e-4, adv_eps=1e-2, start_epoch=1, num_upper_layers=4):
        self.model = model
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.start_epoch = start_epoch
        self.num_upper_layers = num_upper_layers
        self.backup = {}
        self.backup_eps = {}
    
    def _get_target_params(self):
        """Return names of parameters to perturb (upper layers + head)."""
        target_names = set()
        num_layers = self.model.config.num_hidden_layers
        # Upper N layers
        for i in range(num_layers - self.num_upper_layers, num_layers):
            for name, _ in self.model.named_parameters():
                if f'encoder.layer.{i}.' in name or f'layers.{i}.' in name:
                    target_names.add(name)
        # Classifier head
        for name, _ in self.model.named_parameters():
            if 'classifier' in name or 'pooler' in name:
                target_names.add(name)
        return target_names
    
    def attack_step(self, epoch):
        """Apply adversarial perturbation to weights."""
        if epoch < self.start_epoch:
            return
        
        target_names = self._get_target_params()
        e = 1e-6
        
        for name, param in self.model.named_parameters():
            if name not in target_names or param.grad is None:
                continue
            if param.requires_grad and param.grad is not None:
                # Backup original weights
                self.backup[name] = param.data.clone()
                
                # Compute perturbation direction from gradient
                grad_norm = torch.norm(param.grad)
                if grad_norm > e:
                    # Scale perturbation by parameter norm
                    param_norm = torch.norm(param.data)
                    r_at = self.adv_lr * param.grad / (grad_norm + e) * (param_norm + e)
                    
                    # Clip perturbation
                    r_at = torch.clamp(r_at, -self.adv_eps, self.adv_eps)
                    
                    param.data.add_(r_at)
    
    def restore(self):
        """Restore original weights after adversarial step."""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


print('AWP class ready.')
print(f'  Will perturb upper 4 encoder layers + classifier head')
print(f'  Activation after epoch 1 (warmup first)')

AWP class ready.
  Will perturb upper 4 encoder layers + classifier head
  Activation after epoch 1 (warmup first)


## 9C: DeBERTa 5-Fold with LA Loss + AWP

Retrain DeBERTa-v3-base with the improved loss function and AWP regularization.

Changes from Phase 7:
- Focal Loss --> Logit Adjustment Loss (tau=0.5, NO class_weights)
- Added AWP (adv_lr=1e-4, adv_eps=1e-2, start after epoch 1)
- FP16 mixed precision (AMP) for T4 GPU speed
- Slightly increased epochs to 6 (AWP needs more steps to converge)
- LLRD factor 0.9 retained

In [ ]:
# ============================================================
# DEBERTA CONFIG
# ============================================================
DEBERTA_MODEL = 'microsoft/deberta-v3-base'
MAX_LEN = 512
# DeBERTa runs in FP32 (no AMP) so BS=2 is safer on T4 (15GB VRAM)
# ModernBERT can use AMP so it will use BS=4 later
BATCH_SIZE = 2
GRAD_ACCUM = 8     # effective batch = 16
EPOCHS = 6
LR = 2e-5
PATIENCE = 2
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0
N_FOLDS = 5

# AWP config
AWP_LR = 1e-4
AWP_EPS = 1e-2
AWP_START_EPOCH = 1

# LA Loss config
LA_TAU = 0.5        # 0.5 is the standard value (1.0 was too aggressive -> NaN)
LA_SMOOTHING = 0.02  # mild label smoothing for calibration

print(f'DeBERTa model: {DEBERTA_MODEL}')
print(f'Batch: {BATCH_SIZE}x{GRAD_ACCUM} = eff {BATCH_SIZE*GRAD_ACCUM}')
print(f'EPOCHS={EPOCHS}, LR={LR}, PATIENCE={PATIENCE}')
print(f'AWP: lr={AWP_LR}, eps={AWP_EPS}, start_epoch={AWP_START_EPOCH}')
print(f'LA Loss: tau={LA_TAU}, smoothing={LA_SMOOTHING}')
print(f'NOTE: DeBERTa uses FP32 (no AMP) due to disentangled attention FP16 bug')
print(f'NUM_WORKERS={NUM_WORKERS}')

DeBERTa model: microsoft/deberta-v3-base
Batch: 2x8 = eff 16
EPOCHS=6, LR=2e-05, PATIENCE=2
AWP: lr=0.0001, eps=0.01, start_epoch=1
LA Loss: tau=1.0


In [5]:
# ============================================================
# TOKENIZE
# ============================================================
print('Tokenizing for DeBERTa...')
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

deberta_train_enc = deberta_tokenizer(
    list(texts_train), truncation=True, padding=True,
    max_length=MAX_LEN, return_tensors='pt'
)
deberta_test_enc = deberta_tokenizer(
    list(texts_test), truncation=True, padding=True,
    max_length=MAX_LEN, return_tensors='pt'
)

print(f'Train tokens: {deberta_train_enc["input_ids"].shape}')
print(f'Test tokens:  {deberta_test_enc["input_ids"].shape}')

Tokenizing for DeBERTa...
Train tokens: torch.Size([2400, 512])
Test tokens:  torch.Size([600, 512])


In [6]:
# ============================================================
# DATASET & OPTIMIZER
# ============================================================
class PreTokenizedDataset(Dataset):
    def __init__(self, encodings, labels=None, indices=None):
        self.encodings = encodings
        self.labels = labels
        self.indices = indices

    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        real_idx = self.indices[idx] if self.indices is not None else idx
        item = {key: val[real_idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[real_idx], dtype=torch.long)
        return item


def get_optimizer_with_llrd(model, lr, weight_decay=0.01, llrd_factor=0.9):
    """Layer-wise Learning Rate Decay for transformer fine-tuning."""
    opt_params = []
    no_decay = ['bias', 'LayerNorm.weight', 'layernorm.weight']

    # Classifier head: full LR
    head_wd = [p for n, p in model.named_parameters()
        if ('classifier' in n or 'pooler' in n) and not any(nd in n for nd in no_decay)]
    head_nowd = [p for n, p in model.named_parameters()
        if ('classifier' in n or 'pooler' in n) and any(nd in n for nd in no_decay)]
    if head_wd:
        opt_params.append({'params': head_wd, 'lr': lr, 'weight_decay': weight_decay})
    if head_nowd:
        opt_params.append({'params': head_nowd, 'lr': lr, 'weight_decay': 0.0})

    # Encoder layers: decaying LR
    num_layers = model.config.num_hidden_layers
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd_factor ** (num_layers - 1 - layer_idx))
        layer_wd = [p for n, p in model.named_parameters()
            if f'encoder.layer.{layer_idx}.' in n and not any(nd in n for nd in no_decay)]
        layer_nowd = [p for n, p in model.named_parameters()
            if f'encoder.layer.{layer_idx}.' in n and any(nd in n for nd in no_decay)]
        if layer_wd:
            opt_params.append({'params': layer_wd, 'lr': layer_lr, 'weight_decay': weight_decay})
        if layer_nowd:
            opt_params.append({'params': layer_nowd, 'lr': layer_lr, 'weight_decay': 0.0})

    # Embeddings: lowest LR
    emb_lr = lr * (llrd_factor ** num_layers)
    emb_wd = [p for n, p in model.named_parameters()
        if 'embeddings' in n and not any(nd in n for nd in no_decay)]
    emb_nowd = [p for n, p in model.named_parameters()
        if 'embeddings' in n and any(nd in n for nd in no_decay)]
    if emb_wd:
        opt_params.append({'params': emb_wd, 'lr': emb_lr, 'weight_decay': weight_decay})
    if emb_nowd:
        opt_params.append({'params': emb_nowd, 'lr': emb_lr, 'weight_decay': 0.0})

    # Catch-all for remaining parameters
    assigned = set()
    for group in opt_params:
        for p in group['params']:
            assigned.add(id(p))
    remaining_wd = [p for n, p in model.named_parameters()
        if id(p) not in assigned and not any(nd in n for nd in no_decay)]
    remaining_nowd = [p for n, p in model.named_parameters()
        if id(p) not in assigned and any(nd in n for nd in no_decay)]
    if remaining_wd:
        opt_params.append({'params': remaining_wd, 'lr': lr * 0.5, 'weight_decay': weight_decay})
    if remaining_nowd:
        opt_params.append({'params': remaining_nowd, 'lr': lr * 0.5, 'weight_decay': 0.0})

    return torch.optim.AdamW(opt_params)


print('Dataset, LLRD optimizer ready.')

Dataset, LLRD optimizer ready.


In [ ]:
# ============================================================
# 5-FOLD DEBERTA TRAINING WITH LA LOSS + AWP (CUDA-safe)
# ============================================================
def clear_gpu():
    """Clear GPU memory safely on any device."""
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    elif DEVICE.type == 'mps':
        torch.mps.empty_cache()


def train_deberta_fold(fold_idx, train_idx, val_idx):
    """Train DeBERTa for one fold with LA Loss and AWP.
    
    NOTE: DeBERTa-v3's disentangled attention creates FP16 parameter gradients
    under torch.cuda.amp.autocast, which causes 'Attempting to unscale FP16
    gradients' error with GradScaler. Therefore we do NOT use AMP for DeBERTa.
    """
    print(f'\n{"="*60}')
    print(f'DEBERTA FOLD {fold_idx + 1}/{N_FOLDS}')
    print(f'  Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*60}')

    train_ds = PreTokenizedDataset(deberta_train_enc, y_all, train_idx)
    val_ds = PreTokenizedDataset(deberta_train_enc, y_all, val_idx)
    test_ds = PreTokenizedDataset(deberta_test_enc)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))

    model = AutoModelForSequenceClassification.from_pretrained(
        DEBERTA_MODEL, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    model.to(DEVICE)

    # Compute fold-specific class priors for LA Loss
    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    fold_priors = fold_counts / fold_counts.sum()

    # LA Loss -- NO class_weights (LA already handles imbalance)
    criterion = LogitAdjustmentLoss(fold_priors, tau=LA_TAU, label_smoothing=LA_SMOOTHING)
    criterion.to(DEVICE)

    optimizer = get_optimizer_with_llrd(model, lr=LR, llrd_factor=0.9)

    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)

    # AWP
    awp = AWP(model, adv_lr=AWP_LR, adv_eps=AWP_EPS, start_epoch=AWP_START_EPOCH)

    best_f1 = 0
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        t0 = time.time()

        # --- TRAIN (no AMP -- DeBERTa-v3 is incompatible with GradScaler) ---
        model.train()
        total_loss = 0
        valid_steps = 0
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
            desc=f'F{fold_idx+1} E{epoch+1} [Train]', leave=False,
            bar_format='{l_bar}{bar:30}{r_bar}')

        for step, batch in pbar:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            
            outputs = model(**inputs)
            loss = criterion(outputs.logits, labels) / GRAD_ACCUM

            # NaN guard
            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad()
                continue
            
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
            valid_steps += 1

            if (step + 1) % GRAD_ACCUM == 0:
                # AWP: adversarial step on weights
                if epoch >= AWP_START_EPOCH:
                    awp.attack_step(epoch)
                    outputs_adv = model(**inputs)
                    loss_adv = criterion(outputs_adv.logits, labels) / GRAD_ACCUM
                    if not (torch.isnan(loss_adv) or torch.isinf(loss_adv)):
                        loss_adv.backward()
                    awp.restore()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            pbar.set_postfix({'loss': f'{total_loss/max(valid_steps, 1):.4f}'})

        pbar.close()

        # Flush remaining gradients
        if len(train_loader) > 0 and (step + 1) % GRAD_ACCUM != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        avg_loss = total_loss / max(valid_steps, 1)

        # --- VALIDATE ---
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = inputs.pop('labels')
                outputs = model(**inputs)
                all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_f1 = f1_score(all_labels, all_preds, average='macro')
        elapsed = time.time() - t0

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker = '  ** new best **'
        else:
            patience_counter += 1
            marker = f' (patience {patience_counter}/{PATIENCE})'

        awp_status = 'ON' if epoch >= AWP_START_EPOCH else 'OFF'
        print(f'  E{epoch+1} | loss={avg_loss:.4f} | val_f1={val_f1:.4f} | AWP={awp_status} | {elapsed:.0f}s{marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and collect predictions
    model.load_state_dict(best_state)
    model.eval()

    val_logits_list = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            inputs.pop('labels', None)
            outputs = model(**inputs)
            val_logits_list.extend(outputs.logits.cpu().numpy())

    test_logits_list = []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**inputs)
            test_logits_list.extend(outputs.logits.cpu().numpy())

    # Save checkpoint
    torch.save(model.state_dict(), f'checkpoints/deberta_la_fold{fold_idx}.pt')

    del model, optimizer, scheduler, best_state
    clear_gpu()

    return np.array(val_logits_list), np.array(test_logits_list), best_f1


print('DeBERTa training function ready (LA Loss + AWP, FP32).')

DeBERTa training function ready (LA Loss + AWP).


In [8]:
# ============================================================
# RUN 5-FOLD DEBERTA
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

deberta_oof_logits = np.zeros((len(y_all), NUM_LABELS))
deberta_test_logits_sum = np.zeros((len(test_df), NUM_LABELS))
deberta_fold_scores = []

os.makedirs('checkpoints', exist_ok=True)
t_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    val_logits, test_logits, best_f1 = train_deberta_fold(fold_idx, train_idx, val_idx)
    deberta_oof_logits[val_idx] = val_logits
    deberta_test_logits_sum += test_logits
    deberta_fold_scores.append(best_f1)
    print(f'  Fold {fold_idx+1} best val F1: {best_f1:.4f}')
    print(f'  Running time: {(time.time()-t_start)/60:.1f} min')

deberta_test_logits_avg = deberta_test_logits_sum / N_FOLDS
t_deberta = time.time() - t_start

print(f'\n{"="*60}')
print(f'DEBERTA 5-FOLD COMPLETE (LA Loss + AWP)')
print(f'  Fold scores: {[f"{s:.4f}" for s in deberta_fold_scores]}')
print(f'  Mean: {np.mean(deberta_fold_scores):.4f} +/- {np.std(deberta_fold_scores):.4f}')
print(f'  Previous mean (Focal Loss): 0.9426')
print(f'  Total time: {t_deberta/60:.1f} min')
print(f'{"="*60}')


DEBERTA FOLD 1/5
  Train: 1920, Val: 480


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

In [ ]:
# ============================================================
# TEMPERATURE SCALING ON DEBERTA OOF
# ============================================================
logits_t = torch.tensor(deberta_oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all, dtype=torch.long)

best_temp = 1.0
best_nll = float('inf')

for temp in np.arange(0.3, 5.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll:
        best_nll = nll
        best_temp = temp

print(f'Optimal temperature: {best_temp:.2f}')

deberta_oof_probs = torch.softmax(logits_t / best_temp, dim=-1).numpy()
deberta_oof_f1 = f1_score(y_all, deberta_oof_probs.argmax(axis=1), average='macro')

deberta_test_probs = torch.softmax(
    torch.tensor(deberta_test_logits_avg, dtype=torch.float32) / best_temp, dim=-1
).numpy()

print(f'DeBERTa OOF F1 (LA+AWP): {deberta_oof_f1:.4f}')
print(f'Previous DeBERTa OOF F1 (Focal): 0.9428')

print(f'\nOOF per-class report:')
print(classification_report(y_all, deberta_oof_probs.argmax(axis=1),
    target_names=[f'{v}({k})' for k, v in LABEL_MAP.items()]))

# Save
np.save('models/deberta_la_oof_logits.npy', deberta_oof_logits)
np.save('models/deberta_la_oof_probs.npy', deberta_oof_probs)
np.save('models/deberta_la_test_logits.npy', deberta_test_logits_avg)
np.save('models/deberta_la_test_probs.npy', deberta_test_probs)
print('Saved DeBERTa LA+AWP artifacts.')

## 9D: ModernBERT-base 5-Fold Training

A second transformer backbone to provide uncorrelated features.
ModernBERT was pre-trained on 2T tokens including code, giving it
superior structural/formatting awareness -- exactly what we need
for the DeepSeek vs Grok distinction.

Uses LDAM Loss + Deferred Re-Weighting for diversity from DeBERTa.

In [ ]:
# ============================================================
# MODERNBERT CONFIG & TOKENIZE
# ============================================================
MODERN_MODEL = 'answerdotai/ModernBERT-base'
MODERN_LR = 3e-5
MODERN_EPOCHS = 6
# ModernBERT supports AMP fine, so use larger batch on CUDA
MODERN_BATCH_SIZE = 4 if (DEVICE.type == 'cuda' and USE_AMP) else BATCH_SIZE
MODERN_GRAD_ACCUM = 4 if (DEVICE.type == 'cuda' and USE_AMP) else GRAD_ACCUM

print(f'ModernBERT: BS={MODERN_BATCH_SIZE}x{MODERN_GRAD_ACCUM} = eff {MODERN_BATCH_SIZE*MODERN_GRAD_ACCUM}, AMP={USE_AMP}')
print(f'Tokenizing for ModernBERT...')
modern_tokenizer = AutoTokenizer.from_pretrained(MODERN_MODEL)

modern_train_enc = modern_tokenizer(
    list(texts_train), truncation=True, padding=True,
    max_length=MAX_LEN, return_tensors='pt'
)
modern_test_enc = modern_tokenizer(
    list(texts_test), truncation=True, padding=True,
    max_length=MAX_LEN, return_tensors='pt'
)

print(f'Train tokens: {modern_train_enc["input_ids"].shape}')
print(f'Test tokens:  {modern_test_enc["input_ids"].shape}')

In [ ]:
# ============================================================
# MODERNBERT FOLD TRAINING WITH LDAM + DRW (CUDA-safe)
# ============================================================
def get_modernbert_optimizer(model, lr, weight_decay=0.01, llrd_factor=0.9):
    """LLRD for ModernBERT -- uses 'layers.N.' pattern instead of 'encoder.layer.N.'"""
    opt_params = []
    no_decay = ['bias', 'LayerNorm.weight', 'layernorm.weight', 'layer_norm.weight']

    # Classifier head
    head_wd = [p for n, p in model.named_parameters()
        if ('classifier' in n or 'head' in n) and not any(nd in n for nd in no_decay)]
    head_nowd = [p for n, p in model.named_parameters()
        if ('classifier' in n or 'head' in n) and any(nd in n for nd in no_decay)]
    if head_wd:
        opt_params.append({'params': head_wd, 'lr': lr, 'weight_decay': weight_decay})
    if head_nowd:
        opt_params.append({'params': head_nowd, 'lr': lr, 'weight_decay': 0.0})

    # Detect layer pattern
    all_names = [n for n, _ in model.named_parameters()]
    import re as _re
    layer_nums = set()
    for n in all_names:
        m = _re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m:
            layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 12

    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd_factor ** (num_layers - 1 - layer_idx))
        patterns = [f'encoder.layer.{layer_idx}.', f'layers.{layer_idx}.']
        layer_wd = [p for n, p in model.named_parameters()
            if any(pat in n for pat in patterns) and not any(nd in n for nd in no_decay)]
        layer_nowd = [p for n, p in model.named_parameters()
            if any(pat in n for pat in patterns) and any(nd in n for nd in no_decay)]
        if layer_wd:
            opt_params.append({'params': layer_wd, 'lr': layer_lr, 'weight_decay': weight_decay})
        if layer_nowd:
            opt_params.append({'params': layer_nowd, 'lr': layer_lr, 'weight_decay': 0.0})

    # Embeddings
    emb_lr = lr * (llrd_factor ** num_layers)
    emb_wd = [p for n, p in model.named_parameters()
        if 'embeddings' in n and not any(nd in n for nd in no_decay)]
    emb_nowd = [p for n, p in model.named_parameters()
        if 'embeddings' in n and any(nd in n for nd in no_decay)]
    if emb_wd:
        opt_params.append({'params': emb_wd, 'lr': emb_lr, 'weight_decay': weight_decay})
    if emb_nowd:
        opt_params.append({'params': emb_nowd, 'lr': emb_lr, 'weight_decay': 0.0})

    # Remaining
    assigned = set()
    for group in opt_params:
        for p in group['params']:
            assigned.add(id(p))
    rem_wd = [p for n, p in model.named_parameters()
        if id(p) not in assigned and p.requires_grad and not any(nd in n for nd in no_decay)]
    rem_nowd = [p for n, p in model.named_parameters()
        if id(p) not in assigned and p.requires_grad and any(nd in n for nd in no_decay)]
    if rem_wd:
        opt_params.append({'params': rem_wd, 'lr': lr * 0.5, 'weight_decay': weight_decay})
    if rem_nowd:
        opt_params.append({'params': rem_nowd, 'lr': lr * 0.5, 'weight_decay': 0.0})

    return torch.optim.AdamW(opt_params)


def train_modern_fold(fold_idx, train_idx, val_idx):
    """Train ModernBERT for one fold with LDAM + DRW + AMP."""
    print(f'\n{"="*60}')
    print(f'MODERNBERT FOLD {fold_idx + 1}/{N_FOLDS}')
    print(f'  Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*60}')

    train_ds = PreTokenizedDataset(modern_train_enc, y_all, train_idx)
    val_ds = PreTokenizedDataset(modern_train_enc, y_all, val_idx)
    test_ds = PreTokenizedDataset(modern_test_enc)

    train_loader = DataLoader(train_ds, batch_size=MODERN_BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    val_loader = DataLoader(val_ds, batch_size=MODERN_BATCH_SIZE * 2,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    test_loader = DataLoader(test_ds, batch_size=MODERN_BATCH_SIZE * 2,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))

    model = AutoModelForSequenceClassification.from_pretrained(
        MODERN_MODEL, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    model.to(DEVICE)

    # LDAM Loss
    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion = LDAMLoss(fold_counts, max_margin=0.5, drw_epoch=2, total_epochs=MODERN_EPOCHS)
    criterion.to(DEVICE)

    optimizer = get_modernbert_optimizer(model, lr=MODERN_LR, llrd_factor=0.9)

    total_steps = (len(train_loader) // MODERN_GRAD_ACCUM) * MODERN_EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)

    scaler = GradScaler(enabled=USE_AMP)

    best_f1 = 0
    best_state = None
    patience_counter = 0

    for epoch in range(MODERN_EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)

        model.train()
        total_loss = 0
        valid_steps = 0
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
            desc=f'ModF{fold_idx+1} E{epoch+1} [Train]', leave=False,
            bar_format='{l_bar}{bar:30}{r_bar}')

        for step, batch in pbar:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
                loss = criterion(outputs.logits, labels) / MODERN_GRAD_ACCUM
            
            # NaN guard
            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad()
                scaler.update()
                continue
            
            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACCUM
            valid_steps += 1

            if (step + 1) % MODERN_GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            pbar.set_postfix({'loss': f'{total_loss/max(valid_steps, 1):.4f}'})

        pbar.close()

        if len(train_loader) > 0 and (step + 1) % MODERN_GRAD_ACCUM != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        avg_loss = total_loss / max(valid_steps, 1)

        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = inputs.pop('labels')
                with autocast(enabled=USE_AMP):
                    outputs = model(**inputs)
                all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_f1 = f1_score(all_labels, all_preds, average='macro')
        elapsed = time.time() - t0

        drw_status = 'ON' if epoch >= 2 else 'OFF'
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker = '  ** new best **'
        else:
            patience_counter += 1
            marker = f' (patience {patience_counter}/{PATIENCE})'

        print(f'  E{epoch+1} | loss={avg_loss:.4f} | val_f1={val_f1:.4f} | DRW={drw_status} | {elapsed:.0f}s{marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model.eval()

    val_logits_list = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            inputs.pop('labels', None)
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
            val_logits_list.extend(outputs.logits.float().cpu().numpy())

    test_logits_list = []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
            test_logits_list.extend(outputs.logits.float().cpu().numpy())

    torch.save(model.state_dict(), f'checkpoints/modern_fold{fold_idx}.pt')

    del model, optimizer, scheduler, scaler, best_state
    clear_gpu()

    return np.array(val_logits_list), np.array(test_logits_list), best_f1


print('ModernBERT training function ready (LDAM + DRW + AMP).')

In [ ]:
# ============================================================
# RUN 5-FOLD MODERNBERT
# ============================================================
modern_oof_logits = np.zeros((len(y_all), NUM_LABELS))
modern_test_logits_sum = np.zeros((len(test_df), NUM_LABELS))
modern_fold_scores = []

t_start_m = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    val_logits, test_logits, best_f1 = train_modern_fold(fold_idx, train_idx, val_idx)
    modern_oof_logits[val_idx] = val_logits
    modern_test_logits_sum += test_logits
    modern_fold_scores.append(best_f1)
    print(f'  Fold {fold_idx+1} best val F1: {best_f1:.4f}')
    print(f'  Running time: {(time.time()-t_start_m)/60:.1f} min')

modern_test_logits_avg = modern_test_logits_sum / N_FOLDS
t_modern = time.time() - t_start_m

print(f'\n{"="*60}')
print(f'MODERNBERT 5-FOLD COMPLETE (LDAM + DRW)')
print(f'  Fold scores: {[f"{s:.4f}" for s in modern_fold_scores]}')
print(f'  Mean: {np.mean(modern_fold_scores):.4f} +/- {np.std(modern_fold_scores):.4f}')
print(f'  Total time: {t_modern/60:.1f} min')
print(f'{"="*60}')

In [ ]:
# ============================================================
# TEMPERATURE SCALING ON MODERNBERT OOF
# ============================================================
logits_m = torch.tensor(modern_oof_logits, dtype=torch.float32)

best_temp_m = 1.0
best_nll_m = float('inf')
for temp in np.arange(0.3, 5.0, 0.05):
    nll = F.cross_entropy(logits_m / temp, labels_t).item()
    if nll < best_nll_m:
        best_nll_m = nll
        best_temp_m = temp

print(f'ModernBERT optimal temperature: {best_temp_m:.2f}')

modern_oof_probs = torch.softmax(logits_m / best_temp_m, dim=-1).numpy()
modern_oof_f1 = f1_score(y_all, modern_oof_probs.argmax(axis=1), average='macro')

modern_test_probs = torch.softmax(
    torch.tensor(modern_test_logits_avg, dtype=torch.float32) / best_temp_m, dim=-1
).numpy()

print(f'ModernBERT OOF F1: {modern_oof_f1:.4f}')
print(f'\nOOF per-class report:')
print(classification_report(y_all, modern_oof_probs.argmax(axis=1),
    target_names=[f'{v}({k})' for k, v in LABEL_MAP.items()]))

# Save
np.save('models/modern_oof_logits.npy', modern_oof_logits)
np.save('models/modern_oof_probs.npy', modern_oof_probs)
np.save('models/modern_test_logits.npy', modern_test_logits_avg)
np.save('models/modern_test_probs.npy', modern_test_probs)
print('Saved ModernBERT artifacts.')

## 9E: Enhanced Feature Engineering

New features beyond the existing 46:
1. **POS tag bigrams** -- captures syntactic patterns (DeepSeek: formal noun-adj; Grok: pronoun-verb)
2. **Character n-gram SVD** -- 3-5 char n-grams compressed via TruncatedSVD to 200 dims
3. **Burstiness** -- std of per-sentence complexity (humans are bursty, AI is consistent)
4. **Sentence length variance** -- related to burstiness
5. **Function word ratios** -- the/a/an/is/was/etc. patterns differ by model
6. **However/transition patterns** -- Claude-specific fingerprint

In [ ]:
# ============================================================
# ENHANCED FEATURES (self-contained, no src/ dependency)
# ============================================================

def extract_features(texts):
    """Extract 46 linguistic/stylometric features (inlined for Kaggle)."""
    features = []
    for text in tqdm(texts, desc='Base features', leave=False):
        f = {}
        words = text.split()
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]

        f['char_count'] = len(text)
        f['word_count'] = len(words)
        f['sentence_count'] = max(len(sentences), 1)
        f['avg_word_len'] = np.mean([len(w) for w in words]) if words else 0
        f['avg_sentence_len'] = f['word_count'] / f['sentence_count']
        f['max_word_len'] = max([len(w) for w in words]) if words else 0
        f['std_word_len'] = np.std([len(w) for w in words]) if len(words) > 1 else 0

        lower_words = [w.lower() for w in words]
        unique_words = set(lower_words)
        f['unique_word_count'] = len(unique_words)
        f['type_token_ratio'] = len(unique_words) / max(len(words), 1)
        word_freq = Counter(lower_words)
        f['hapax_ratio'] = sum(1 for v in word_freq.values() if v == 1) / max(len(unique_words), 1)
        freq_spectrum = Counter(word_freq.values())
        N = len(lower_words)
        M2 = sum(i * i * freq_spectrum[i] for i in freq_spectrum)
        f['yules_k'] = 10000 * (M2 - N) / (N * N) if N > 1 else 0

        f['upper_ratio'] = sum(1 for c in text if c.isupper()) / max(len(text), 1)
        f['digit_ratio'] = sum(1 for c in text if c.isdigit()) / max(len(text), 1)
        f['space_ratio'] = sum(1 for c in text if c.isspace()) / max(len(text), 1)
        f['alpha_ratio'] = sum(1 for c in text if c.isalpha()) / max(len(text), 1)

        f['punct_ratio'] = sum(1 for c in text if c in string.punctuation) / max(len(text), 1)
        f['comma_ratio'] = text.count(',') / max(len(words), 1)
        f['period_ratio'] = text.count('.') / max(len(words), 1)
        f['exclamation_ratio'] = text.count('!') / max(len(words), 1)
        f['question_ratio'] = text.count('?') / max(len(words), 1)
        f['semicolon_ratio'] = text.count(';') / max(len(words), 1)
        f['colon_ratio'] = text.count(':') / max(len(words), 1)
        f['quote_count'] = text.count('"') + text.count("'")
        f['paren_count'] = text.count('(') + text.count(')')
        f['dash_count'] = text.count('-') + text.count('\u2014')

        f['double_letter_ratio'] = len(re.findall(r'(.)\1', text.lower())) / max(len(words), 1)
        f['short_word_ratio'] = sum(1 for w in words if len(w) <= 2) / max(len(words), 1)
        f['long_word_ratio'] = sum(1 for w in words if len(w) > 10) / max(len(words), 1)
        f['very_long_word_ratio'] = sum(1 for w in words if len(w) > 15) / max(len(words), 1)
        consonant_clusters = re.findall(r'[bcdfghjklmnpqrstvwxyz]{3,}', text.lower())
        f['consonant_cluster_ratio'] = len(consonant_clusters) / max(len(words), 1)
        repeated_chars = re.findall(r'(.)\1{2,}', text)
        f['repeated_char_count'] = len(repeated_chars)

        f['paragraph_count'] = text.count('\n\n') + 1
        f['newline_count'] = text.count('\n')
        f['starts_with_upper'] = int(text[0].isupper()) if text else 0
        f['ends_with_period'] = int(text.rstrip().endswith('.')) if text else 0
        f['bullet_count'] = len(re.findall(r'^\s*[-*\u2022]\s', text, re.MULTILINE))
        f['numbered_list_count'] = len(re.findall(r'^\s*\d+[.):]\s', text, re.MULTILINE))

        text_lower = text.lower()
        ai_phrases = ['in conclusion', 'furthermore', 'moreover', 'additionally',
            'it is worth noting', 'it is important to', 'on the other hand',
            'in summary', 'to summarize', 'in essence', 'ultimately',
            'delve', 'crucial', 'comprehensive', 'leverage', 'facilitate',
            'robust', 'streamline', 'innovative', 'cutting-edge', 'paradigm',
            'holistic', 'synergy', 'encompass', 'multifaceted', 'nuanced']
        f['ai_phrase_count'] = sum(1 for p in ai_phrases if p in text_lower)
        transitions = ['however', 'therefore', 'nonetheless', 'nevertheless',
            'consequently', 'meanwhile', 'subsequently', 'accordingly', 'hence', 'thus', 'thereby']
        f['transition_count'] = sum(1 for t in transitions if t in text_lower)
        first_person = re.findall(r'\b(i|me|my|mine|myself|we|us|our|ours|ourselves)\b', text_lower)
        f['first_person_ratio'] = len(first_person) / max(len(words), 1)
        contractions = re.findall(r"\b\w+'\w+\b", text)
        f['contraction_ratio'] = len(contractions) / max(len(words), 1)

        def _count_syllables(word):
            word = word.lower().rstrip('e')
            return max(1, len(re.findall(r'[aeiouy]+', word)))
        total_syllables = sum(_count_syllables(w) for w in words) if words else 1
        f['flesch_reading_ease'] = 206.835 - 1.015 * f['avg_sentence_len'] - 84.6 * (total_syllables / max(len(words), 1))
        f['flesch_kincaid_grade'] = 0.39 * f['avg_sentence_len'] + 11.8 * (total_syllables / max(len(words), 1)) - 15.59
        f['ari'] = 4.71 * (f['char_count'] / max(len(words), 1)) + 0.5 * f['avg_sentence_len'] - 21.43

        char_freq = Counter(text.lower())
        total_chars = sum(char_freq.values())
        char_probs = [c / total_chars for c in char_freq.values()]
        f['char_entropy'] = -sum(p * np.log2(p) for p in char_probs if p > 0)
        word_probs = [c / len(lower_words) for c in word_freq.values()] if lower_words else [1]
        f['word_entropy'] = -sum(p * np.log2(p) for p in word_probs if p > 0)

        features.append(f)
    return pd.DataFrame(features)


def extract_advanced_features(texts):
    """Extract additional stylometric features beyond the base 46."""
    function_words = {'the', 'a', 'an', 'is', 'was', 'are', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'shall', 'should',
        'may', 'might', 'must', 'can', 'could', 'of', 'in', 'to', 'for', 'with',
        'on', 'at', 'from', 'by', 'about', 'as', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'and', 'but', 'or', 'nor',
        'not', 'so', 'yet', 'both', 'either', 'neither', 'each', 'every', 'all',
        'this', 'that', 'these', 'those', 'it', 'its', 'he', 'she', 'they', 'them',
        'his', 'her', 'their', 'which', 'who', 'whom', 'what', 'where', 'when'}
    records = []
    for text in tqdm(texts, desc='Advanced features', leave=False):
        f = {}
        words = text.split()
        lower_words = [w.lower().strip(string.punctuation) for w in words]
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        n_func = sum(1 for w in lower_words if w in function_words)
        f['function_word_ratio'] = n_func / max(len(words), 1)
        sent_lens = [len(s.split()) for s in sentences]
        f['sentence_len_std'] = np.std(sent_lens) if len(sent_lens) > 1 else 0
        f['sentence_len_cv'] = (np.std(sent_lens) / max(np.mean(sent_lens), 1)) if len(sent_lens) > 1 else 0
        if len(sent_lens) > 1:
            m, s = np.mean(sent_lens), np.std(sent_lens)
            f['burstiness'] = (s - m) / (s + m) if (s + m) > 0 else 0
        else:
            f['burstiness'] = 0
        f['however_capital_ratio'] = len(re.findall(r'\bHowever\b', text)) / max(len(sentences), 1)
        starters = [s.split()[0].lower() if s.split() else '' for s in sentences]
        f['starter_diversity'] = len(set(starters)) / max(len(sentences), 1)
        punct_chars = [c for c in text if c in string.punctuation]
        if punct_chars:
            punct_freq = Counter(punct_chars)
            total_p = sum(punct_freq.values())
            probs = [c / total_p for c in punct_freq.values()]
            f['punct_entropy'] = -sum(p * np.log2(p) for p in probs if p > 0)
        else:
            f['punct_entropy'] = 0
        sent_complexities = []
        for s in sentences:
            sw = s.split()
            if sw:
                sent_complexities.append(np.mean([len(w) for w in sw]))
        f['complexity_std'] = np.std(sent_complexities) if len(sent_complexities) > 1 else 0
        f['comma_period_ratio'] = text.count(',') / max(text.count('.'), 1)
        passive_patterns = len(re.findall(r'\b(was|were|been|being|is|are)\s+\w+ed\b', text.lower()))
        f['passive_voice_ratio'] = passive_patterns / max(len(sentences), 1)
        paragraphs = text.split('\n\n')
        para_lens = [len(p.split()) for p in paragraphs if p.strip()]
        f['para_len_std'] = np.std(para_lens) if len(para_lens) > 1 else 0
        content_words = [w for w in lower_words if w not in function_words and len(w) > 2]
        f['lexical_density'] = len(content_words) / max(len(words), 1)
        conjunctions = {'and', 'but', 'or', 'nor', 'for', 'yet', 'so', 'because',
            'although', 'while', 'whereas', 'unless', 'since', 'until'}
        f['conjunction_ratio'] = sum(1 for w in lower_words if w in conjunctions) / max(len(words), 1)
        records.append(f)
    return pd.DataFrame(records)


print('Extracting base 46 features...')
train_feats = extract_features(texts_train)
test_feats = extract_features(texts_test)

print('Extracting advanced features...')
train_adv = extract_advanced_features(texts_train)
test_adv = extract_advanced_features(texts_test)

train_feats_full = pd.concat([train_feats, train_adv], axis=1)
test_feats_full = pd.concat([test_feats, test_adv], axis=1)

print(f'Base features: {train_feats.shape[1]}')
print(f'Advanced features: {train_adv.shape[1]}')
print(f'Total: {train_feats_full.shape[1]}')

In [ ]:
# ============================================================
# CHARACTER N-GRAM SVD
# ============================================================
print('Building character n-gram TF-IDF + SVD...')

# Char n-grams (3-5) compressed via TruncatedSVD
tfidf_char_svd = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=50000, sublinear_tf=True,
    min_df=2, max_df=0.95
)
X_char_train = tfidf_char_svd.fit_transform(texts_train)
X_char_test = tfidf_char_svd.transform(texts_test)

svd = TruncatedSVD(n_components=200, random_state=SEED)
X_char_svd_train = svd.fit_transform(X_char_train)
X_char_svd_test = svd.transform(X_char_test)

explained = svd.explained_variance_ratio_.sum()
print(f'Char n-gram TF-IDF: {X_char_train.shape}')
print(f'SVD (200 components): explains {explained:.1%} of variance')

# Also build word n-gram TF-IDF + SVD
tfidf_word_svd = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    max_features=50000, sublinear_tf=True,
    min_df=2, max_df=0.95
)
X_word_train = tfidf_word_svd.fit_transform(texts_train)
X_word_test = tfidf_word_svd.transform(texts_test)

svd_word = TruncatedSVD(n_components=200, random_state=SEED)
X_word_svd_train = svd_word.fit_transform(X_word_train)
X_word_svd_test = svd_word.transform(X_word_test)

explained_w = svd_word.explained_variance_ratio_.sum()
print(f'Word n-gram TF-IDF: {X_word_train.shape}')
print(f'SVD (200 components): explains {explained_w:.1%} of variance')

In [ ]:
# ============================================================
# COMBINE ALL FEATURES FOR CLASSICAL MODELS
# ============================================================
# Scale handcrafted features
scaler = StandardScaler()
X_feats_train = scaler.fit_transform(train_feats_full.values)
X_feats_test = scaler.transform(test_feats_full.values)

# Combine: handcrafted + char SVD + word SVD
X_all_train = np.hstack([X_feats_train, X_char_svd_train, X_word_svd_train])
X_all_test = np.hstack([X_feats_test, X_char_svd_test, X_word_svd_test])

print(f'Combined feature matrix:')
print(f'  Handcrafted: {X_feats_train.shape[1]}')
print(f'  Char SVD: {X_char_svd_train.shape[1]}')
print(f'  Word SVD: {X_word_svd_train.shape[1]}')
print(f'  Total: {X_all_train.shape}')

In [ ]:
# ============================================================
# CLASSICAL MODELS: SVC + LR + XGB + LGB (5-FOLD OOF)
# ============================================================
# Also train on the FULL feature space (raw TF-IDF, not just SVD)
# for SVC/LR which can handle high-dim sparse matrices

X_sparse_train = sparse.hstack([X_word_train, X_char_train,
    sparse.csr_matrix(X_feats_train)]).tocsr()
X_sparse_test = sparse.hstack([X_word_test, X_char_test,
    sparse.csr_matrix(X_feats_test)]).tocsr()

print(f'Sparse feature matrix for SVC/LR: {X_sparse_train.shape}')

classical_models = [
    ('SVC', CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000, random_state=SEED),
        cv=3, method='sigmoid'
    ), X_sparse_train, X_sparse_test),
    ('LR', LogisticRegression(
        C=2.0, class_weight='balanced', max_iter=5000, solver='lbfgs', random_state=SEED
    ), X_sparse_train, X_sparse_test),
    ('XGB', xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        min_child_weight=3, random_state=SEED,
        tree_method='hist', eval_metric='mlogloss',
        early_stopping_rounds=50
    ), X_all_train, X_all_test),
    ('LGB', lgb.LGBMClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        min_child_weight=3, random_state=SEED,
        class_weight='balanced', verbose=-1,
        n_jobs=1
    ), X_all_train, X_all_test),
]

classical_oof_probs = {}
classical_test_probs = {}

for name, model_template, X_tr, X_te in classical_models:
    print(f'\n--- {name} 5-fold ---')
    oof_probs = np.zeros((len(y_all), NUM_LABELS))
    test_probs_sum = np.zeros((len(test_df), NUM_LABELS))
    fold_f1s = []

    for fold, (tr_idx, vl_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
        m = clone(model_template)
        
        if name == 'XGB':
            m.fit(X_tr[tr_idx], y_all[tr_idx],
                eval_set=[(X_tr[vl_idx], y_all[vl_idx])],
                verbose=False)
        else:
            m.fit(X_tr[tr_idx], y_all[tr_idx])
        
        if hasattr(m, 'predict_proba'):
            fold_probs = m.predict_proba(X_tr[vl_idx])
        else:
            # Fallback: one-hot from predictions
            preds = m.predict(X_tr[vl_idx])
            fold_probs = np.zeros((len(vl_idx), NUM_LABELS))
            fold_probs[np.arange(len(vl_idx)), preds] = 1.0
        
        oof_probs[vl_idx] = fold_probs
        fold_f1 = f1_score(y_all[vl_idx], fold_probs.argmax(axis=1), average='macro')
        fold_f1s.append(fold_f1)
        
        if hasattr(m, 'predict_proba'):
            test_probs_sum += m.predict_proba(X_te) / N_FOLDS
        else:
            preds = m.predict(X_te)
            fold_test = np.zeros((len(test_df), NUM_LABELS))
            fold_test[np.arange(len(test_df)), preds] = 1.0 / N_FOLDS
            test_probs_sum += fold_test
    
    oof_f1 = f1_score(y_all, oof_probs.argmax(axis=1), average='macro')
    print(f'  OOF F1: {oof_f1:.4f} (folds: {[f"{s:.4f}" for s in fold_f1s]})')
    
    classical_oof_probs[name] = oof_probs
    classical_test_probs[name] = test_probs_sum

# Save
for name in classical_oof_probs:
    np.save(f'models/v3_{name.lower()}_oof_probs.npy', classical_oof_probs[name])
    np.save(f'models/v3_{name.lower()}_test_probs.npy', classical_test_probs[name])

print('\nSaved all classical model artifacts.')

## 9F: Multi-Model Ensemble + Per-Class Threshold Optimization

Now we have 6 models producing OOF probabilities:
1. DeBERTa-v3 (LA + AWP) -- semantic features
2. ModernBERT (LDAM + DRW) -- structural features
3. SVC -- high-dim sparse features
4. LR -- high-dim sparse features
5. XGBoost -- dense combined features
6. LightGBM -- dense combined features

Strategy:
1. Weight search across all 6 models
2. Per-class threshold optimization on the blended probabilities
3. Stacking meta-learner (Ridge) as alternative

In [ ]:
# ============================================================
# ENSEMBLE WEIGHT OPTIMIZATION (6 MODELS)
# ============================================================
all_oof = {
    'DeBERTa': deberta_oof_probs,
    'ModernBERT': modern_oof_probs,
    'SVC': classical_oof_probs['SVC'],
    'LR': classical_oof_probs['LR'],
    'XGB': classical_oof_probs['XGB'],
    'LGB': classical_oof_probs['LGB'],
}
all_test = {
    'DeBERTa': deberta_test_probs,
    'ModernBERT': modern_test_probs,
    'SVC': classical_test_probs['SVC'],
    'LR': classical_test_probs['LR'],
    'XGB': classical_test_probs['XGB'],
    'LGB': classical_test_probs['LGB'],
}

# Individual model OOF scores
print('Individual model OOF F1:')
for name, probs in all_oof.items():
    f1 = f1_score(y_all, probs.argmax(axis=1), average='macro')
    print(f'  {name:12s}: {f1:.4f}')

# Scipy optimization for 6 weights (Nelder-Mead)
model_names = list(all_oof.keys())
oof_arrays = [all_oof[n] for n in model_names]

def neg_macro_f1(weights):
    """Negative macro F1 for minimization."""
    weights = np.abs(weights)
    weights = weights / weights.sum()
    blend = sum(w * p for w, p in zip(weights, oof_arrays))
    return -f1_score(y_all, blend.argmax(axis=1), average='macro')

# Try multiple initializations
best_result = None
best_f1_opt = 0

init_configs = [
    [0.35, 0.25, 0.15, 0.10, 0.08, 0.07],  # DeBERTa heavy
    [0.30, 0.30, 0.15, 0.10, 0.08, 0.07],  # Balanced transformers
    [0.40, 0.20, 0.15, 0.10, 0.08, 0.07],  # DeBERTa heavier
    [0.25, 0.25, 0.20, 0.15, 0.08, 0.07],  # More classical
    [1/6]*6,  # Uniform
]

for init in init_configs:
    result = minimize(neg_macro_f1, init, method='Nelder-Mead',
        options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-6})
    if -result.fun > best_f1_opt:
        best_f1_opt = -result.fun
        best_result = result

opt_weights = np.abs(best_result.x)
opt_weights = opt_weights / opt_weights.sum()

print(f'\nOptimized 6-model weights:')
for name, w in zip(model_names, opt_weights):
    print(f'  {name:12s}: {w:.4f}')
print(f'\nBlended OOF F1: {best_f1_opt:.4f}')

In [ ]:
# ============================================================
# PER-CLASS THRESHOLD OPTIMIZATION
# ============================================================
blend_oof = sum(w * p for w, p in zip(opt_weights, oof_arrays))

test_arrays = [all_test[n] for n in model_names]
blend_test = sum(w * p for w, p in zip(opt_weights, test_arrays))

def optimize_thresholds_v2(probs, labels, n_classes=6, n_passes=5, steps=100):
    """Advanced per-class threshold optimization.
    
    Uses multipliers: pred = argmax(probs * multipliers)
    Greedy coordinate descent with fine grid.
    """
    best_mult = np.ones(n_classes)
    best_f1 = f1_score(labels, probs.argmax(axis=1), average='macro')
    
    for pass_idx in range(n_passes):
        improved = False
        # Iterate classes in order of increasing sample count (minority first)
        class_order = np.argsort(np.bincount(labels, minlength=n_classes))
        
        for cls in class_order:
            for mult in np.linspace(0.3, 3.0, steps):
                trial = best_mult.copy()
                trial[cls] = mult
                preds = (probs * trial).argmax(axis=1)
                f1 = f1_score(labels, preds, average='macro')
                if f1 > best_f1 + 1e-7:
                    best_f1 = f1
                    best_mult = trial.copy()
                    improved = True
        
        if not improved:
            break
    
    return best_mult, best_f1


print('Optimizing per-class thresholds...')
thresh_mult, thresh_f1 = optimize_thresholds_v2(blend_oof, y_all)

print(f'\nPer-class multipliers:')
for c in range(NUM_LABELS):
    n = (y_all == c).sum()
    print(f'  {LABEL_MAP[c]:10s} (n={n:4d}): mult={thresh_mult[c]:.4f}')

print(f'\nBefore thresholds: OOF F1 = {best_f1_opt:.4f}')
print(f'After thresholds:  OOF F1 = {thresh_f1:.4f}')
print(f'Improvement: {thresh_f1 - best_f1_opt:+.4f}')

In [ ]:
# ============================================================
# STACKING META-LEARNER (RIDGE + MLP)
# ============================================================
from sklearn.neural_network import MLPClassifier

def build_rich_meta_features(prob_dict):
    """Build rich meta-features from all model probabilities."""
    features = []
    names = list(prob_dict.keys())
    
    # 1. Raw probabilities (N_models x 6)
    for name in names:
        features.append(prob_dict[name])
    
    # 2. Per-model confidence (N_models x 1)
    for name in names:
        features.append(prob_dict[name].max(axis=1, keepdims=True))
    
    # 3. Per-model entropy (N_models x 1)
    for name in names:
        p = prob_dict[name]
        ent = -np.sum(p * np.log(np.clip(p, 1e-10, 1.0)), axis=1, keepdims=True)
        features.append(ent)
    
    # 4. Pairwise max disagreement (top-3 models pairwise)
    top_models = ['DeBERTa', 'ModernBERT', 'SVC']
    for i, n1 in enumerate(top_models):
        for n2 in top_models[i+1:]:
            diff = np.abs(prob_dict[n1] - prob_dict[n2]).max(axis=1, keepdims=True)
            features.append(diff)
    
    # 5. Transformer agreement
    d_pred = prob_dict['DeBERTa'].argmax(axis=1)
    m_pred = prob_dict['ModernBERT'].argmax(axis=1)
    features.append((d_pred == m_pred).astype(float).reshape(-1, 1))
    
    # 6. Mean probabilities across all models
    mean_probs = np.mean([prob_dict[n] for n in names], axis=0)
    features.append(mean_probs)
    
    # 7. Std of probabilities across models per class
    std_probs = np.std([prob_dict[n] for n in names], axis=0)
    features.append(std_probs)
    
    return np.hstack(features)


meta_X_oof = build_rich_meta_features(all_oof)
meta_X_test = build_rich_meta_features(all_test)
print(f'Meta-features: {meta_X_oof.shape[1]}')

# Try multiple meta-learners
meta_configs = [
    ('LR_C1', LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)),
    ('LR_C2', LogisticRegression(C=2.0, class_weight='balanced', max_iter=5000, random_state=SEED)),
    ('LR_C5', LogisticRegression(C=5.0, class_weight='balanced', max_iter=5000, random_state=SEED)),
    ('MLP_small', MLPClassifier(hidden_layer_sizes=(64,), alpha=0.01, max_iter=500,
        random_state=SEED, early_stopping=True, validation_fraction=0.15)),
    ('MLP_medium', MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.01, max_iter=500,
        random_state=SEED, early_stopping=True, validation_fraction=0.15)),
]

best_meta_name = None
best_meta_f1 = 0
best_meta_test_probs = None

for meta_name, meta_template in meta_configs:
    skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_probs = np.zeros((len(y_all), NUM_LABELS))
    test_probs_sum = np.zeros((len(test_df), NUM_LABELS))
    fold_f1s = []

    for fold, (tr_idx, vl_idx) in enumerate(skf_meta.split(meta_X_oof, y_all)):
        m = clone(meta_template)
        m.fit(meta_X_oof[tr_idx], y_all[tr_idx])
        fold_probs = m.predict_proba(meta_X_oof[vl_idx])
        oof_probs[vl_idx] = fold_probs
        fold_f1s.append(f1_score(y_all[vl_idx], fold_probs.argmax(axis=1), average='macro'))
        test_probs_sum += m.predict_proba(meta_X_test) / 5

    oof_f1 = f1_score(y_all, oof_probs.argmax(axis=1), average='macro')
    print(f'{meta_name:12s}: OOF F1={oof_f1:.4f} (folds: {[f"{s:.4f}" for s in fold_f1s]})')

    if oof_f1 > best_meta_f1:
        best_meta_f1 = oof_f1
        best_meta_name = meta_name
        best_meta_test_probs = test_probs_sum.copy()

print(f'\nBest meta-learner: {best_meta_name} with OOF F1={best_meta_f1:.4f}')

In [ ]:
# ============================================================
# DISTRIBUTION-AWARE SOFT PSEUDO-LABELING
# ============================================================
print('Distribution-Aware Soft Pseudo-Labeling...')

# Use the optimized blend as soft labels for test set
soft_test_labels = blend_test.copy()

# Probability trimming: zero out any class probability < 0.10
soft_test_labels[soft_test_labels < 0.10] = 0.0
# Re-normalize
row_sums = soft_test_labels.sum(axis=1, keepdims=True)
soft_test_labels = soft_test_labels / np.maximum(row_sums, 1e-10)

# Select high-confidence samples per class (class-balanced quota)
hard_test_preds = blend_test.argmax(axis=1)
hard_test_conf = blend_test.max(axis=1)

# Take top-K per class (class-balanced selection)
K_PER_CLASS = 15  # conservative quota
selected_indices = []
selected_labels = []

for cls in range(NUM_LABELS):
    cls_mask = hard_test_preds == cls
    cls_indices = np.where(cls_mask)[0]
    cls_conf = hard_test_conf[cls_mask]
    
    # Sort by confidence, take top K
    n_select = min(K_PER_CLASS, len(cls_indices))
    if n_select > 0:
        top_k = np.argsort(cls_conf)[-n_select:]
        selected_indices.extend(cls_indices[top_k])
        selected_labels.extend([cls] * n_select)

selected_indices = np.array(selected_indices)
selected_labels = np.array(selected_labels)

print(f'Selected {len(selected_indices)} pseudo-labeled samples (class-balanced):')
for cls in range(NUM_LABELS):
    n = (selected_labels == cls).sum()
    print(f'  {LABEL_MAP[cls]:10s}: {n} samples')

# Show confidence stats for selected
sel_conf = hard_test_conf[selected_indices]
print(f'\nConfidence of selected samples:')
print(f'  Mean: {sel_conf.mean():.4f}, Min: {sel_conf.min():.4f}, Max: {sel_conf.max():.4f}')

In [ ]:
# ============================================================
# COMPARISON AND BEST SUBMISSION SELECTION
# ============================================================
print('='*60)
print('PHASE 9 RESULTS COMPARISON')
print('='*60)

# All methods
methods = {
    'DeBERTa (LA+AWP)': deberta_oof_f1,
    'ModernBERT (LDAM+DRW)': modern_oof_f1,
}
for name in classical_oof_probs:
    f1 = f1_score(y_all, classical_oof_probs[name].argmax(axis=1), average='macro')
    methods[name] = f1

methods['6-model weighted blend'] = best_f1_opt
methods['Weighted blend + thresholds'] = thresh_f1
methods['Stacking meta-learner'] = best_meta_f1

print(f'\nAll methods:')
for name, f1 in sorted(methods.items(), key=lambda x: -x[1]):
    print(f'  {name:35s}: {f1:.4f}')

print(f'\nPrevious best OOF: 0.9591')
print(f'Previous best LB:  0.92170')

In [ ]:
# ============================================================
# GENERATE SUBMISSIONS
# ============================================================
os.makedirs('submissions', exist_ok=True)

def write_submission(preds, path, name):
    n = len(preds)
    lines = ['ID,LABEL'] + [f'{i},{int(preds[i])}' for i in range(n)]
    with open(path, 'w', newline='\n') as f:
        f.write('\n'.join(lines) + '\n')
    dist = dict(zip(*np.unique(preds.astype(int), return_counts=True)))
    print(f'  {name}: {dist}')

# Submission 1: 6-model weighted blend (no thresholds)
blend_preds = blend_test.argmax(axis=1)
write_submission(blend_preds,
    'submissions/08_v3_6model_blend.csv', '6-model blend')

# Submission 2: 6-model blend + thresholds
thresh_preds = (blend_test * thresh_mult).argmax(axis=1)
write_submission(thresh_preds,
    'submissions/09_v3_blend_thresholds.csv', 'Blend + thresholds')

# Submission 3: Stacking meta-learner
stack_preds = best_meta_test_probs.argmax(axis=1)
write_submission(stack_preds,
    'submissions/10_v3_stacking.csv', 'Stacking meta-learner')

# Submission 4: DeBERTa-only (LA+AWP)
deberta_only_preds = deberta_test_probs.argmax(axis=1)
write_submission(deberta_only_preds,
    'submissions/11_v3_deberta_la.csv', 'DeBERTa LA+AWP only')

print(f'\nAll submissions generated.')

In [ ]:
# ============================================================
# SUBMIT TO KAGGLE (works both on Kaggle and locally)
# ============================================================
COMPETITION = 'malto-recruitment-hackathon'

# On Kaggle, kaggle CLI is pre-installed. Locally, set up path.
if not os.path.exists('/kaggle'):
    os.environ['PATH'] += ':/Users/aliivaezii/Library/Python/3.9/bin'

submissions_to_send = [
    ('submissions/08_v3_6model_blend.csv',
     f'V3: 6-model blend (DeBERTa-LA+ModernBERT+SVC+LR+XGB+LGB), OOF={best_f1_opt:.4f}'),
    ('submissions/09_v3_blend_thresholds.csv',
     f'V3: 6-model blend + per-class thresholds, OOF={thresh_f1:.4f}'),
    ('submissions/10_v3_stacking.csv',
     f'V3: 6-model stacking ({best_meta_name}), OOF={best_meta_f1:.4f}'),
    ('submissions/11_v3_deberta_la.csv',
     f'V3: DeBERTa LA+AWP 5-fold, OOF={deberta_oof_f1:.4f}'),
]

for path, msg in submissions_to_send:
    if os.path.exists(path):
        print(f'Submitting {os.path.basename(path)}...')
        cmd = f'kaggle competitions submit -c {COMPETITION} -f {path} -m "{msg}"'
        result = os.popen(cmd).read()
        print(f'  {result.strip()}')
        time.sleep(5)

print('\nWaiting 30s for scores...')
time.sleep(30)

result = os.popen(f'kaggle competitions submissions -c {COMPETITION}').read()
print(result)

In [ ]:
# ============================================================
# SAVE PHASE 9 CONFIG
# ============================================================
config = {
    'version': 'v3_phase9',
    'improvements': [
        f'Logit Adjustment Loss (tau={LA_TAU}, smoothing={LA_SMOOTHING})',
        'Adversarial Weight Perturbation (AWP)',
        'ModernBERT-base as second transformer',
        'Enhanced features: 58 handcrafted + char/word SVD (400 dims)',
        'Distribution-aware soft pseudo-labeling',
        'Per-class threshold optimization (fine grid)',
        'Scipy Nelder-Mead weight optimization (6 models)',
        f'Mixed precision (AMP={USE_AMP})',
    ],
    'deberta': {
        'loss': f'LogitAdjustment (tau={LA_TAU}, smoothing={LA_SMOOTHING})',
        'awp': f'lr={AWP_LR}, eps={AWP_EPS}, start_epoch={AWP_START_EPOCH}',
        'fold_scores': [float(s) for s in deberta_fold_scores],
        'mean_f1': float(np.mean(deberta_fold_scores)),
        'oof_f1': float(deberta_oof_f1),
        'temperature': float(best_temp),
    },
    'modernbert': {
        'loss': 'LDAM (max_margin=0.5, drw_epoch=2)',
        'fold_scores': [float(s) for s in modern_fold_scores],
        'mean_f1': float(np.mean(modern_fold_scores)),
        'oof_f1': float(modern_oof_f1),
        'temperature': float(best_temp_m),
    },
    'ensemble': {
        'weights': {n: float(w) for n, w in zip(model_names, opt_weights)},
        'blend_oof_f1': float(best_f1_opt),
        'thresh_oof_f1': float(thresh_f1),
        'thresh_multipliers': thresh_mult.tolist(),
        'stacking_meta': best_meta_name,
        'stacking_oof_f1': float(best_meta_f1),
    },
    'previous_best': {
        'oof_f1': 0.9591,
        'public_lb': 0.92170,
    },
}

with open('models/phase9_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved: models/phase9_config.json')
print(json.dumps(config, indent=2))

In [ ]:
# ============================================================
# PER-CLASS ANALYSIS + CONFUSION MATRIX
# ============================================================
# Use the best OOF method
best_oof_preds = (blend_oof * thresh_mult).argmax(axis=1)

print('Per-class OOF F1 -- Best ensemble (blend + thresholds):')
per_class = f1_score(y_all, best_oof_preds, average=None)
for c in range(NUM_LABELS):
    n = (y_all == c).sum()
    print(f'  {LABEL_MAP[c]:10s} (n={n:4d}): F1={per_class[c]:.4f}')
print(f'\nMacro F1 = {per_class.mean():.4f}')

# Confusion matrix
cm = confusion_matrix(y_all, best_oof_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Phase 9 confusion
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[0],
    xticklabels=[LABEL_MAP[i] for i in range(6)],
    yticklabels=[LABEL_MAP[i] for i in range(6)])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Phase 9 (LA+AWP+ModernBERT)\nMacro F1 = {per_class.mean():.4f}', fontweight='bold')

# Model contribution
names_short = ['DeB', 'Mod', 'SVC', 'LR', 'XGB', 'LGB']
axes[1].barh(names_short, opt_weights, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#795548'])
axes[1].set_xlabel('Weight')
axes[1].set_title('Ensemble Weights', fontweight='bold')
for i, w in enumerate(opt_weights):
    axes[1].text(w + 0.01, i, f'{w:.3f}', va='center')

plt.tight_layout()
plt.savefig('figures/phase9_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFigure saved: figures/phase9_results.png')